### Obx events mapping
- 0 is analog visual
- 2 is trial start
- 3 is frames
- 4 is left port
- 5 is center port
- 6 is right port

In [ ]:
from labdata.schema import (
    EphysRecording,
    File,
    Dataset,
    DatasetEvents,
    StreamSync,
    Session,
)
import numpy as np
from tqdm import tqdm

In [ ]:
subject = "GRB058"
session = "20260224_152424"
sess_query = Session() & f'subject_name = "{subject}"' & f'session_name = "{session}"'

In [ ]:
# find a file
sessions = (
    EphysRecording().proj()
    & f'subject_name = "{subject}"'
    & f'session_name = "{session}"'
)

for ses in tqdm(sessions):
    if len(DatasetEvents & ses & 'stream_name = "obx"'):
        continue
    paths = (
        File() & (Dataset.DataFiles & ses & 'file_path LIKE "%.obx.%"')
    ).check_if_files_local()[0]
    from spks.spikeglx_utils import load_spikeglx_binary
    from spks import unpack_npix_sync

    dat, meta = load_spikeglx_binary(paths[0])
    onsets, offsets = unpack_npix_sync(dat[:, -2])  # unpack the syncs for the behavior
    clock = unpack_npix_sync(dat[:, -1])[0][6]  # unpack the clock
    srate = meta["sRateHz"]
    # insert the events
    stream = "obx"
    DatasetEvents().insert(
        [dict(ses, stream_name=stream)],
        allow_direct_insert=True,
        skip_duplicates=True,
    )
    digital_events = []
    for o in onsets.keys():
        digital_events.append(
            dict(
                ses,
                stream_name=stream,
                event_name=int(o),
                event_timestamps=(onsets[o] / srate).astype(np.float32),
            )
        )
    digital_events.append(
        dict(
            ses,
            stream_name=stream,
            event_name="sync",
            event_timestamps=(clock / srate).astype(np.float32),
        )
    )
    DatasetEvents.Digital().insert(
        digital_events, allow_direct_insert=True, skip_duplicates=True
    )

    StreamSync().insert(
        [
            dict(
                ses,
                stream_name=s,
                event_name=6,
                clock_dataset=ses["dataset_name"],
                clock_stream="nidq",
                clock_stream_event=6,
            )
            for s in ["imec0"]
        ],
        skip_duplicates=True,
    )
    StreamSync().insert(
        [
            dict(
                ses,
                stream_name="obx",
                event_name="sync",
                clock_dataset=ses["dataset_name"],
                clock_stream="nidq",
                clock_stream_event=6,
            )
        ],
        skip_duplicates=True,
    )